# Doğal Dil İşleme + Karar Bilimi (Natural Language Processing + Decision Science)

👩🏻‍🏫 Bu görevde şunları bir araya getireceğiz:
* 🗣 Doğal Dil İşleme (Natural Language Processing)
* 📊 Karar Bilimi (Decision Science)

🎯 Amaç, Olist üzerindeki ürünlerin ve satıcıların **olumsuz (kötü) yorumlarını** anlamaktır.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
# Data Manipulation
import numpy as np
import pandas as pd
pd.set_option("display.max_columns",None)

# Machine Learning
from sklearn.pipeline import make_pipeline

# Language Processing
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
import nltk
import string
import unidecode as unidecode

# Vectorizers and NLP Models
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

🕵🏻‍♂️ Olist’in CEO’su [Tiago Dalvi](https://www.linkedin.com/in/tiagodalvi/)’nin senden yorumları okuyup anlamanı istediğini hayal et.

- Müşteriler siparişlerini **1**, **2** veya **3** puanla değerlendirdiklerinde ne söylediler?
- En sık karşılaşılan olumsuz yorumlar neler?
    - En kötü puanlanan ürünler hakkında?
    - En kötü puanlanan satıcılar hakkında?


## (0) Kurulum 🔨

Öncelikle, Olist incelemeleriyle ilgili tüm bilgileri içeren DataFrame'i yükleyeceğiz!

In [4]:
df = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/olist_reviews.csv")

In [5]:
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n"
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok..."


## (2) Metin Temizleme (Text Cleaning)

🧹 `cleaning(sentence)` işlevini oluşturun ve yorumlara uygulayın. **NLTK'da Portekizce lemmatizer bulunmadığını unutmayın** (genellikle bulunmaz, ancak `nltk.stem.RSLPStemmer` kök bulucu vardır).

In [6]:
import string
import unidecode
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

nltk.download('rslp')
nltk.download('stopwords')

def cleaning(sentence):
    # Aksanları kaldır (ç, ã, õ gibi Portekizce özel karakterler)
    sentence = unidecode.unidecode(sentence)
    # Küçük harfe çevir
    sentence = sentence.lower()
    # Noktalama işaretlerini kaldır
    for punctuation in string.punctuation:
        sentence = sentence.replace(punctuation, '')
    # Sayıları kaldır
    sentence = ''.join(char for char in sentence if not char.isdigit())
    # Tokenization
    tokens = word_tokenize(sentence, language='portuguese')
    # Stopwords kaldır
    stop_words = set(stopwords.words('portuguese'))
    tokens = [word for word in tokens if word not in stop_words]
    # Portekizce'de lemmatizer yok, RSLPStemmer kullan
    stemmer = RSLPStemmer()
    tokens = [stemmer.stem(word) for word in tokens]
    return ' '.join(tokens)

[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\BÜŞRA\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\BÜŞRA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
df["full_review_cleaned"] = df["full_review"].apply(cleaning)
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned
0,e64fb393e7b32834bb789ff8bb30750e,37,5,658677c97b385a9be170737859d3511b,ferramentas_jardim,Recebi bem antes do prazo estipulado.,receb bem ant praz estipul
1,f7c4243c7fe1938f181bec41a392bdeb,100,5,8e6bfb81e283fa7e4f11123a3fb894f1,esporte_lazer,Parabéns lojas lannister adorei comprar pela ...,parab loj lannist ador compr internet segur pr...
2,8670d52e15e00043ae7de4c01cc2fe06,174,4,b9bf720beb4ab3728760088589c62129,eletroportateis,recomendo aparelho eficiente. no site a marca ...,recom aparelh efici sit marc aparelh impress d...
3,4b49719c8a200003f700d3d986ea1a19,45,4,9d6f15f95d01e79bd1349cc208361f09,beleza_saude,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",pouc travandopel val ta boa
4,3948b09f7c818e2d86c9a546758b2335,56,5,e51478e7e277a83743b6f9991dbfa3fb,informatica_acessorios,"Super recomendo Vendedor confiável, produto ok...",sup recom vend confia produt ok entreg ant praz


In [8]:
df.columns

Index(['review_id', 'length_review', 'review_score', 'order_id',
       'product_category_name', 'full_review', 'full_review_cleaned'],
      dtype='object')

## (3) Kötü yorumların analizi

### (3.1) Düşük inceleme puanlarına sahip veri kümesi

😱 1 ile 3 arasında puan alan yorumların oranı nedir? 

In [9]:
ratio = (df["review_score"] <= 3).mean()
ratio

np.float64(0.26717937368595773)

🕵🏻‍♂️ Bu yorumlara odaklanalım...

In [10]:
df = df[df["review_score"] <= 3].reset_index(drop=True)
df.shape

(9658, 7)

### (3.2) Vektörleştirme

🔡 ➡️ 🔢 Metinlerini vektörleştir.

- **Bigram**’leri (iki kelimelik ifadeler) mutlaka hesaba kat.
- Çok sık geçen kelimeleri çıkarmak için `max_df = 0.75` ayarla.
- Spoiler uyarısı: Sonunda **20.000+** kelimeye ulaşacaksın…  
  Bu challenge için sadece `max_features = 5000` ile sınırla.

In [11]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000)
X = vectorizer.fit_transform(df["full_review_cleaned"])

### (3.3) LDA

🕵🏻‍♂️ LDA'yı uyarlayın:
- `n_components = 3` seçin
- *.transform()* ile Konuların Belge Karışımını gösterin
- *.components_* ile Konu Karışımını gösterin

In [12]:
from sklearn.decomposition import LatentDirichletAllocation

lda_model = LatentDirichletAllocation(n_components=3, random_state=42)
lda_model.fit(X)

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0


#### Belge Karışımı (Konular için)

In [13]:
document_topic_mixture = lda_model.transform(X)
document_topic_mixture

array([[0.02386012, 0.17260461, 0.80353527],
       [0.02396957, 0.72259683, 0.2534336 ],
       [0.68019535, 0.02044581, 0.29935884],
       ...,
       [0.24431166, 0.74242689, 0.01326145],
       [0.2713605 , 0.71159036, 0.01704914],
       [0.64586499, 0.02475428, 0.32938073]], shape=(9658, 3))

👉 Her inceleme için en önemli konuyu rapor edelim

In [14]:
df["most_important_topic"] = document_topic_mixture.argmax(axis=1)
df.head()

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic
0,e233e51d11511bf30e568c76360ace52,97,1,548df2c6e5f089574614894bca78acf5,eletronicos,recebi somente 1 controle Midea Split ESTILO....,receb soment control mide split estil falt con...,2
1,8b230a1373c6dc4bd867099fda1d7039,60,3,071251fe3b3493294536f03737a8a679,ferramentas_jardim,Eu comprei duas unidades e só recebi uma e ag...,compr dua unidad so receb agor fac,1
2,cb2fc3e5711b5ae85e0491ee18af63ed,72,3,34e6d418f368f8079ae152bc178bc66a,beleza_saude,"Produto bom, porém o que veio para mim não co...",produt bom por vei mim nao condiz fot anunci,0
3,60c714ed14cef913944a3147094a4742,36,1,9ac05114800f02bfaa783bd76842dbe2,moveis_decoracao,"Produto muito inferior, mal acabado.",produt inferi mal acab,0
4,0bd4dcc4f6c4621baf37f73495cad8c4,16,3,a11cd01ac67beef7e8bf09740d8a35c1,esporte_lazer,Entrega no prazo,entreg praz,0


#### Konu Karışımı (Kelimeler için)

In [15]:
lda_model.components_

array([[10.30050483, 40.16343134,  4.10295579, ...,  0.33677924,
         3.1771681 , 12.87239019],
       [ 2.14708253,  5.31432075,  1.38812057, ...,  0.35379705,
        21.42200138,  0.33899497],
       [ 0.55241264,  6.52224791,  2.50892364, ...,  8.30942371,
         0.40083052,  3.78861484]], shape=(3, 5000))

#### Konular

🎁 Size bazı yardımcı fonksiyonlar sağladık:
- `topic_word`: Tek bir konu (topic) için en önemli kelimeleri ve ağırlıklarını döndürür
- `print_topics`: LDA tarafından bulunan farklı konuları, en önemli kelimeleriyle birlikte yazdırır

In [16]:
def topic_word(vectorizer, model, topic, topwords, with_weights = True):
    topwords_indexes = topic.argsort()[:-topwords - 1:-1]
    if with_weights == True:
        topwords = [(vectorizer.get_feature_names_out()[i], round(topic[i],2)) for i in topwords_indexes]
    if with_weights == False:
        topwords = [vectorizer.get_feature_names_out()[i] for i in topwords_indexes]
    return topwords

In [17]:
def print_topics(vectorizer, model, topwords):
    for idx, topic in enumerate(model.components_):
        print("-"*20)
        print("Topic %d:" % (idx))
        print(topic_word(vectorizer, model, topic, topwords))


🕵🏻‍♂️ Konuları en çok kullanılan kelimelerle birlikte yazdırın:

In [18]:
print_topics(vectorizer, lda_model, topwords=10)

--------------------
Topic 0:
[('nao', np.float64(2988.46)), ('produt', np.float64(2569.69)), ('entreg', np.float64(1270.63)), ('praz', np.float64(708.23)), ('bom', np.float64(649.19)), ('qual', np.float64(593.28)), ('gost', np.float64(483.5)), ('cheg', np.float64(451.11)), ('produt nao', np.float64(430.88)), ('recom', np.float64(406.47))]
--------------------
Topic 1:
[('nao', np.float64(2498.48)), ('receb', np.float64(2458.74)), ('produt', np.float64(1486.08)), ('compr', np.float64(1412.13)), ('entreg', np.float64(1136.34)), ('nao receb', np.float64(788.24)), ('apen', np.float64(632.43)), ('ped', np.float64(585.05)), ('receb produt', np.float64(564.95)), ('so', np.float64(551.79))]
--------------------
Topic 2:
[('vei', np.float64(1797.21)), ('produt', np.float64(1749.23)), ('compr', np.float64(1125.39)), ('outr', np.float64(613.25)), ('err', np.float64(537.9)), ('cheg', np.float64(531.55)), ('falt', np.float64(437.85)), ('produt vei', np.float64(437.43)), ('ped', np.float64(430.48))

🇧🇷 Burada biraz Brezilya Portekizcesi kelimeler var:
- _cadeiras = chairs_
- _produto = product_
- _recomendo = recommend (não recomendo == not recommend)_
- _bom = good_
- _comprei = bought_
- _veio = came_
- _errado = wrong_
- _gostaria = I would like to..._
- _duas = two_
- _nao = not_
- _entregue = delivered_
- _pecas = part_
- _ainda = yet_
- _recebi = received_

👉 Bir konuyla ilişkili en popüler kelimeleri göster

In [19]:
topic_word_mixture = [
    topic_word(vectorizer, lda_model, topic, topwords=10, with_weights=False)
    for topic in lda_model.components_
]
topic_word_mixture

[['nao',
  'produt',
  'entreg',
  'praz',
  'bom',
  'qual',
  'gost',
  'cheg',
  'produt nao',
  'recom'],
 ['nao',
  'receb',
  'produt',
  'compr',
  'entreg',
  'nao receb',
  'apen',
  'ped',
  'receb produt',
  'so'],
 ['vei',
  'produt',
  'compr',
  'outr',
  'err',
  'cheg',
  'falt',
  'produt vei',
  'ped',
  'so']]

In [20]:
df["most_important_words"] = df["most_important_topic"].apply(lambda i: topic_word_mixture[i])

In [21]:
df[["review_id",
        "review_score",
        "product_category_name",
        "full_review_cleaned",
        "most_important_topic",
        "most_important_words"]
      ].head()

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
0,e233e51d11511bf30e568c76360ace52,1,eletronicos,receb soment control mide split estil falt con...,2,"[vei, produt, compr, outr, err, cheg, falt, pr..."
1,8b230a1373c6dc4bd867099fda1d7039,3,ferramentas_jardim,compr dua unidad so receb agor fac,1,"[nao, receb, produt, compr, entreg, nao receb,..."
2,cb2fc3e5711b5ae85e0491ee18af63ed,3,beleza_saude,produt bom por vei mim nao condiz fot anunci,0,"[nao, produt, entreg, praz, bom, qual, gost, c..."
3,60c714ed14cef913944a3147094a4742,1,moveis_decoracao,produt inferi mal acab,0,"[nao, produt, entreg, praz, bom, qual, gost, c..."
4,0bd4dcc4f6c4621baf37f73495cad8c4,3,esporte_lazer,entreg praz,0,"[nao, produt, entreg, praz, bom, qual, gost, c..."


## (3.4) Pipeline Tf-Idf ve LDA

In [22]:
from sklearn import set_config
set_config("diagram")

🔨 Önceki Vectorizer ve LDA'yı birbirine bağlayan bir Pipeline oluşturun.

Temizlenmiş metinlere uyarlayın.

In [23]:
from sklearn.pipeline import make_pipeline

pipeline = make_pipeline(
    CountVectorizer(ngram_range=(1, 2), max_df=0.75, max_features=5000),
    LatentDirichletAllocation(n_components=3, random_state=42)
)
pipeline.fit(df["full_review_cleaned"])

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('countvectorizer', ...), ('latentdirichletallocation', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float in range [0.0, 1.0] or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float, the parameter represents a proportion of documents, integerabsolute counts.This parameter is ignored if vocabulary is not None.",0.75
,"max_features max_features: int, default=NoneIf not None, build a vocabulary that only consider the top`max_features` ordered by term frequency across the corpus.Otherwise, all features are used.This parameter is ignored if vocabulary is not None.",5000
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' i

💡 `pipeline.components_` ile bileşenlere erişmeye çalışırsanız, Pipeline'da `components_` olmadığı için bu YÜRÜMEZ. Ancak, LDA'ya erişmek için `pipeline._final_estimator` kullanabilirsiniz. Ve buradan konulara erişebilirsiniz!

In [24]:
pipeline._final_estimator

,"n_components n_components: int, default=10Number of topics... versionchanged:: 0.19 ``n_topics`` was renamed to ``n_components``",3
,"random_state random_state: int, RandomState instance or None, default=NonePass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",42
,"doc_topic_prior doc_topic_prior: float, default=NonePrior of document topic distribution `theta`. If the value is None,defaults to `1 / n_components`.In [1]_, this is called `alpha`.",None
,"topic_word_prior topic_word_prior: float, default=NonePrior of topic word distribution `beta`. If the value is None, defaultsto `1 / n_components`.In [1]_, this is called `eta`.",None
,"learning_method learning_method: {'batch', 'online'}, default='batch'Method used to update `_component`. Only used in :meth:`fit` method.In general, if the data size is large, the online update will be muchfaster than the batch update.Valid options:- 'batch': Batch variational Bayes method. Use all training data in each EM update. Old `components_` will be overwritten in each iteration.- 'online': Online variational Bayes method. In each EM update, use mini-batch of training data to update the ``components_`` variable incrementally. The learning rate is controlled by the ``learning_decay`` and the ``learning_offset`` parameters... versionchanged:: 0.20 The default learning method is now ``""batch""``.",'batch'
,"learning_decay learning_decay: float, default=0.7It is a parameter that control learning rate in the online learningmethod. The value should be set between (0.5, 1.0] to guaranteeasymptotic convergence. When the value is 0.0 and batch_size is``n_samples``, the update method is same as batch learning. In theliterature, this is called kappa.",0.7
,"learning_offset learning_offset: float, default=10.0A (positive) parameter that downweights early iterations in onlinelearning. It should be greater than 1.0. In the literature, this iscalled tau_0.",10.0
,"max_iter max_iter: int, default=10The maximum number of passes over the training data (aka epochs).It only impacts the behavior in the :meth:`fit` method, and not the:meth:`partial_fit` method.",10
,"batch_size batch_size: int, default=128Number of documents to use in each EM iteration. Only used in onlinelearning.",128
,"evaluate_every evaluate_every: int, default=-1How often to evaluate perplexity. Only used in `fit` method.set it to 0 or negative number to not evaluate perplexity intraining at all. Evaluating perplexity can help you check convergencein training process, but it will also increase total training time.Evaluating perplexity in every iteration might increase training timeup to two-fold.",-1
,"total_samples total_samples: int, default=1e6Total number of documents. Only used in the :meth:`partial_fit` method.",1000000.0


In [25]:
pipeline._final_estimator.components_

array([[10.30050483, 40.16343134,  4.10295579, ...,  0.33677924,
         3.1771681 , 12.87239019],
       [ 2.14708253,  5.31432075,  1.38812057, ...,  0.35379705,
        21.42200138,  0.33899497],
       [ 0.55241264,  6.52224791,  2.50892364, ...,  8.30942371,
         0.40083052,  3.78861484]], shape=(3, 5000))

Pipeline ile **Belge Karışımı**:

In [26]:
pipeline.transform(df["full_review_cleaned"])

array([[0.02386012, 0.17260461, 0.80353527],
       [0.02396957, 0.72259683, 0.2534336 ],
       [0.68019535, 0.02044581, 0.29935884],
       ...,
       [0.24431166, 0.74242689, 0.01326145],
       [0.2713605 , 0.71159036, 0.01704914],
       [0.64586499, 0.02475428, 0.32938073]], shape=(9658, 3))

Pipeline ile **Konu Karışımı**:

In [27]:
pipeline._final_estimator.components_

array([[10.30050483, 40.16343134,  4.10295579, ...,  0.33677924,
         3.1771681 , 12.87239019],
       [ 2.14708253,  5.31432075,  1.38812057, ...,  0.35379705,
        21.42200138,  0.33899497],
       [ 0.55241264,  6.52224791,  2.50892364, ...,  8.30942371,
         0.40083052,  3.78861484]], shape=(3, 5000))

## (4) 🎁 Ürün Kategorileri

### (4.1) Ürün kategorilerine göre gruplandırma

📈 Veri kümesini `product_category_name` ile gruplandırın ve performanslarını inceleyin.

In [28]:
# Performansa göre ürün kategorileri - sayı, ortalama, medyan ve standart sapmaya bakın
product_categories = df.groupby(by = 'product_category_name').agg({
        'review_score': ["count", "mean", "median", "std"]
    })

# Analiz için belirli bir süreden daha az satılan ürünleri kaldırma
cutoff = 50
product_categories = product_categories[product_categories[("review_score", "count")] > cutoff]

# Ürün kategorilerini performansa göre sıralama
product_categories = product_categories.sort_values(by = [('review_score', 'mean'),
                                                          ('review_score', 'std')],
                                                    ascending = [False, True])
product_categories

review_score                           
                                         count      mean median       std
product_category_name                                                    
consoles_games                              92  1.956522    2.0  0.924787
fashion_bolsas_e_acessorios                155  1.948387    2.0  0.917319
malas_acessorios                            66  1.924242    2.0  0.949666
papelaria                                  155  1.922581    2.0  0.922556
casa_construcao                             66  1.909091    2.0  0.836242
utilidades_domesticas                      586  1.889078    2.0  0.888473
cool_stuff                                 298  1.885906    2.0  0.906618
relogios_presentes                         589  1.881154    2.0  0.905088
pet_shop                                   147  1.857143    2.0  0.883641
moveis_escritorio                          261  1.854406    1.0  0.920843
cama_mesa_banho                           1221  1.850942    2.0  0.890235
esporte_lazer                              626  1.846645    2.0  0.899136
casa_conforto                               57  1.842105    2.0  0.902169
beleza_saude                               716  1.840782    1.0  0.908275
ferramentas_jardim                         318  1.817610    1.0  0.904635
telefonia                                  488  1.815574    2.0  0.875636
moveis_decoracao                           743  1.802153    1.0  0.888620
brinquedos                                 298  1.788591    1.0  0.890808
bebes                                      244  1.786885    1.0  0.877046
eletronicos                                248  1.778226    1.0  0.883502
construcao_ferramentas_construcao           71  1.774648    1.0  0.881511
automotivo                                 370  1.770270    1.0  0.870275
perfumaria                                 235  1.723404    1.0  0.879543
informatica_acessorios                     797  1.708908    1.0  0.867089
eletrodomesticos                            84  1.630952    1.0  0.875087
eletroportateis                             54  1.518519    1.0  0.770708

### (4.2) En kötü ürün kategorileri

👎 *Ortalama değerlendirme puanı* açısından en kötü beş kategoriyi `worst_products` adlı bir değişkene kaydedin.

In [29]:
worst_products = product_categories.tail(5).sort_values(by = [("review_score", "count")],
                                                       ascending = False)
worst_products

review_score                           
                              count      mean median       std
product_category_name                                         
informatica_acessorios          797  1.708908    1.0  0.867089
automotivo                      370  1.770270    1.0  0.870275
perfumaria                      235  1.723404    1.0  0.879543
eletrodomesticos                 84  1.630952    1.0  0.875087
eletroportateis                  54  1.518519    1.0  0.770708

👇 Yalnızca `worst_products` öğelerini içeren bir `worst_products_review` DataFrame oluşturun.

In [30]:
worst_products_reviews = df[df.product_category_name.isin(worst_products.index)]
worst_products_reviews[["review_id",
                        "review_score",
                        "product_category_name",
                        "full_review_cleaned",
                        "most_important_topic",
                        "most_important_words"]
      ]

,review_id,review_score,product_category_name,full_review_cleaned,most_important_topic,most_important_words
5,ff722b4c68783a4459a3adb9bb4e1d0d,3,informatica_acessorios,produt cheg pc nao consegu reconhec port usb,1,"[nao, receb, produt, compr, entreg, nao receb,..."
14,5f938e5f5f2e9a75710b54feeb9ea610,1,eletrodomesticos,medi pec nao serv,0,"[nao, produt, entreg, praz, bom, qual, gost, c..."
19,6b341682ab39af9fa00d72d7388c903b,3,automotivo,not compr produt corret cap intern outr,2,"[vei, produt, compr, outr, err, cheg, falt, pr..."
24,b736ff4204044e49e39062584d06fa74,1,eletroportateis,loj anunc produt entreg outr,2,"[vei, produt, compr, outr, err, cheg, falt, pr..."
29,6f885e2fd69412264d9e74e805175d53,1,informatica_acessorios,receb bem rap por vei outr produt nao ped mode...,2,"[vei, produt, compr, outr, err, cheg, falt, pr..."
...,...,...,...,...,...,...
9633,e9f0bde9a98ff79305964e9033ed8cd2,1,informatica_acessorios,produt cheg falt pess condico,2,"[vei, produt, compr, outr, err, cheg, falt, pr..."
9634,44380245bf1a49875542ea0e7f216986,1,informatica_acessorios,nao receb produt aind assim receb produt avali...,1,"[nao, receb, produt, compr, entreg, nao receb,..."
9642,970ae7c9df61910a0a30c38670db7ae5,3,informatica_acessorios,probl past multius pra notebook ped cor vermel...,2,"[vei, produt, compr, outr, err, cheg, falt, pr..."
9644,504403288b387de1c95a193909ba5dfd,3,perfumaria,cheg bem rapid produt razoa,0,"[nao, produt, entreg, praz, bom, qual, gost, c..."


### (4.3). En kötü ürünler için konular

❓ En kötü ürünlerin konuları nelerdir? ❓

In [31]:
worst_products_reviews["most_important_topic"].value_counts()

most_important_topic
0    627
1    499
2    414
Name: count, dtype: int64

In [32]:
bad_frequency = list(worst_products_reviews["most_important_topic"].value_counts().index)
bad_frequency

[0, 1, 2]

In [33]:
[topic_word_mixture[i] for i in bad_frequency]

[['nao',
  'produt',
  'entreg',
  'praz',
  'bom',
  'qual',
  'gost',
  'cheg',
  'produt nao',
  'recom'],
 ['nao',
  'receb',
  'produt',
  'compr',
  'entreg',
  'nao receb',
  'apen',
  'ped',
  'receb produt',
  'so'],
 ['vei',
  'produt',
  'compr',
  'outr',
  'err',
  'cheg',
  'falt',
  'produt vei',
  'ped',
  'so']]

## (5) 🎁 Satıcılar...

* En kötü satıcılar tarafından ne tür ürünler satıldı?
* En kötü satıcılar için başlıca yorumlar nelerdir?

### (5.1) En kötü satıcılar

In [36]:
from olist.seller import Seller
sellers = Seller().get_training_data()
sellers.columns

d:\DOSYALARIM\workintech gh\S18D1-S-Data-Lda\S18D1-S-Data-bad_reviews_analysis\olist\seller.py:68: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(delay_to_logistic_partner)\
d:\DOSYALARIM\workintech gh\S18D1-S-Data-Lda\S18D1-S-Data-bad_reviews_analysis\olist\seller.py:73: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(order_wait_time)\


Index(['seller_id', 'seller_city', 'seller_state', 'delay_to_carrier',
       'wait_time', 'date_first_sale', 'date_last_sale', 'months_on_olist',
       'share_of_five_stars', 'share_of_one_stars', 'review_score',
       'cost_of_reviews', 'n_orders', 'quantity', 'quantity_per_order',
       'sales', 'revenues', 'profits'],
      dtype='object')

👇 En kötü satan 10 ürünü seçin ve bunları `worst_sellers` adlı bir değişkene kaydedin.

In [37]:
worst_sellers = sellers[["seller_id", "review_score", "profits"]].sort_values(
    by = "profits",
    ascending = True).head(10)
worst_sellers

,seller_id,review_score,profits
769,6560211a19b47992c3666cc44a7e94c0,3.909406,-26269.517
453,1f50f920176fa81dab994f9023523100,3.982402,-25436.079
1132,7c67e1448b00f6e969d365cea6b010ab,3.348208,-23977.611
2358,4a3ca9315b744ce9f8e9374361493884,3.803931,-22972.708
1357,cc419e0650a3c5ba77189a1882b7556a,4.069575,-19121.158
945,ea8482cd71df3c1969d7b9473ff13abc,3.953216,-17882.248
2603,1025f0e2d44d7041d6cf58b6550e0bfa,3.849755,-16273.145
315,8b321bb669392f5163d04c59e235e066,3.995069,-15906.431
1213,d2374cbcbb3ca4ab1086534108cc3ab7,3.636364,-13837.608
2687,1835b56ce799e6a4dc4eddc053f04066,3.593863,-11425.579


### (5.2) En kötü satıcılar tarafından satılan ürünler

In [39]:
from olist.product import Product

products = Product().get_training_data()[["product_id", "category"]]
products

,product_id,category
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares
...,...,...
31479,a0b7d5a992ccda646f2d34e418fff5a0,furniture_decor
31480,bf4538d88321d0fd4412a93c974510e6,construction_tools_lights
31481,9a7c6041fa9592d9d9ef6cfe62a71f8c,bed_bath_table
31482,83808703fc0706a22e264b9d75f04a2e,computers_accessories


❓ En kötü satıcılar tarafından satılan ürün türleri nelerdir? ❓

In [44]:
sellers_product_category = data["order_items"].merge(products,
                                             on = "product_id", how = "left")[["seller_id", "category"]]

sellers_product_category

,seller_id,category
0,48436dade18ac8b2bce089ec2a041202,cool_stuff
1,dd7ddc04e1b6c2c614352b383efe2d36,pet_shop
2,5b51032eddd242adc84c38acab88f23d,furniture_decor
3,9d7a1d34a5052409006425275ba1c2b4,perfumery
4,df560393f3a51e74553ab94004ba5c87,garden_tools
...,...,...
112645,b8bc237ba3788b23da09c0f1f3a3288c,housewares
112646,f3c38ab652836d21de61fb8314b69182,computers_accessories
112647,c3cfdc648177fdbbbb35635a37472c53,sports_leisure
112648,2b3e4a2a3ea8e01938cabda2a3e5cc79,computers_accessories


In [43]:
from olist.data import Olist

data = Olist().get_data()

In [45]:
sellers_product_category.groupby("seller_id").count()

,category
seller_id,
0015a82c2db000af6aaaf3ae2ecb0532,3
001cca7ae9ae17fb1caed9dfb1094831,239
001e6ad469a905060d959994f1b41e4f,0
002100f778ceb8431b7a1020ff7ab48f,55
003554e2dce176b5555353e4f3555ac8,0
...,...
ffcfefa19b08742c5d315f2791395ee5,0
ffdd9f82b9a447f6f8d4b91554cc7dd3,20
ffeee66ac5d5a62fe688b9d26f83f534,14


### (5.3) En kötü satıcılar için kategoriler ve konular

🎁 İşte bazı kullanışlı işlevler:
- Bir satıcı tarafından satılan ürün kategorilerini göstermek için `focus_seller(seller_id)`
- Bir satıcı için en sık kullanılan konuların en popüler kelimelerini göstermek için `bad_reviews_seller`

In [46]:
def focus_seller(seller_id):
    return sellers_product_category[sellers_product_category.seller_id == seller_id].value_counts()

In [47]:
bad_reviews_sellers = worst_products_reviews.merge(data["order_items"])
bad_reviews_sellers.head(3)

,review_id,length_review,review_score,order_id,product_category_name,full_review,full_review_cleaned,most_important_topic,most_important_words,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ff722b4c68783a4459a3adb9bb4e1d0d,66,3,b8702f50a689621ab21ef200b55a5b8d,informatica_acessorios,"Produto chegou, mas meu PC não conseguiu reco...",produt cheg pc nao consegu reconhec port usb,1,"[nao, receb, produt, compr, entreg, nao receb,...",1,e76438822c17316d276c27e52e588931,5656537e588803a555b8eb41f07a944b,2018-01-10 22:07:28,24.89,17.63
1,5f938e5f5f2e9a75710b54feeb9ea610,17,1,d9ff0185a300435402f043124f530274,eletrodomesticos,Médio A peça não serviu,medi pec nao serv,0,"[nao, produt, entreg, praz, bom, qual, gost, c...",1,082175ee534338e4fd967076a6f848e5,d91fb3b7d041e83b64a00a3edfb37e4f,2018-05-10 16:54:16,52.00,18.24
2,6b341682ab39af9fa00d72d7388c903b,66,3,fd65651f855608ad1aa52ff4a741c31a,automotivo,Nota3 Comprar um produto correto na capa mas i...,not compr produt corret cap intern outr,2,"[vei, produt, compr, outr, err, cheg, falt, pr...",1,1dce3fd38c13a3e74fe5000f33858442,72c5da29406b4234927b81855e7b64f6,2018-05-07 05:30:57,229.99,19.49


In [48]:
def bad_reviews_seller(bad_reviews_sellers, seller_id):
    mask = (bad_reviews_sellers.seller_id == seller_id)
    temp = bad_reviews_sellers[mask]
    if len(temp) > 0: # satıcı kötü yorumlar veri çerçevesinde görünüyorsa
        most_frequent_topic_seller = list(temp.most_important_topic.value_counts().head(1).index)[0]
        return topic_word_mixture[most_frequent_topic_seller]

❓Bu en az satan ürünlerin her biri için en sık kullanılan ürün kategorilerini ve kelimeleri gösterin ❓

In [49]:
for worst_seller in worst_sellers["seller_id"]:
    print("-"*50)
    print(f"Focusing on the seller #{worst_seller}...")
    print(focus_seller(worst_seller))
    print(bad_reviews_seller(bad_reviews_sellers, worst_seller))


--------------------------------------------------
Focusing on the seller #6560211a19b47992c3666cc44a7e94c0...
seller_id                         category                 
6560211a19b47992c3666cc44a7e94c0  watches_gifts                1624
                                  fashion_bags_accessories      340
                                  audio                          32
                                  perfumery                      13
                                  computers_accessories          12
                                  sports_leisure                  7
                                  construction_tools_safety       1
Name: count, dtype: int64
['nao', 'receb', 'produt', 'compr', 'entreg', 'nao receb', 'apen', 'ped', 'receb produt', 'so']
--------------------------------------------------
Focusing on the seller #1f50f920176fa81dab994f9023523100...
seller_id                         category              
1f50f920176fa81dab994f9023523100  garden_tools              188

🏁 Tebrikler. NLP'nin bazı temellerini (Ön İşleme + Vektörleştirme + NB/LDA) öğrendiniz ve bu yeni “uzmanlığı” Karar Bilimi ile birleştirdik.

💾 `git add / commit / push` yapmayı unutmayın.